# NVIDIA Nemotron Unsloth Training — V125

> **Unsloth Framework**: Uses `unsloth/Nemotron-3-Nano-30B-A3B` for training with gate_proj/x_proj LoRA on Mamba layers.
> Includes Unsloth→HF PEFT adapter conversion pipeline for submission.
>
> **Key differences from HF PEFT (V123)**:
> - Unsloth splits Mamba `in_proj` into `gate_proj` + `x_proj` → separate LoRA → more parameters on Mamba
> - Unsloth fuses MoE expert weights (w1/w2) → efficient 3D LoRA
> - `alpha=32, dropout=0.0` aligned with 0.85 solution
> - Same boxed loss weight 5x strategy

In [1]:
import subprocess, sys, os, glob

# ================================================================
# Strategy: All offline packages from hastws/sft-offline-packages
#   - Contains: unsloth, unsloth_zoo, trl, tyro, hf_transfer,
#     nest_asyncio, annotated_doc
#   - Other deps (transformers, peft, datasets, accelerate, torch,
#     bitsandbytes, triton, sentencepiece) already in Kaggle env
#   - No dependency on third-party datasets
# ================================================================

OFFLINE_DIRS = [
    "/kaggle/input/sft-offline-packages",
    "/kaggle/input/datasets/hastws/sft-offline-packages",
]

offline_dir = None
for d in OFFLINE_DIRS:
    if os.path.isdir(d):
        whl_files = [f for f in os.listdir(d) if f.endswith('.whl')]
        if whl_files:
            offline_dir = d
            print(f"Found offline packages at: {d} ({len(whl_files)} wheels)")
            break

if offline_dir:
    whls = sorted(glob.glob(os.path.join(offline_dir, "*.whl")))
    print(f"Installing {len(whls)} wheels with --no-deps...")
    for whl in whls:
        name = os.path.basename(whl)
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps", whl],
            capture_output=True, text=True, timeout=120
        )
        status = "OK" if r.returncode == 0 else "FAIL"
        print(f"  {status}: {name}")
        if r.returncode != 0 and "unsloth" in name.lower():
            print(f"    ERROR: {r.stderr[-200:]}")
else:
    print("WARNING: Offline packages not found, trying online install...")
    for pkg in ["unsloth", "unsloth_zoo", "trl"]:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps", pkg],
            capture_output=True, text=True, timeout=300
        )
        print(f"  {'OK' if r.returncode == 0 else 'FAIL'}: {pkg}")

# --- flash_attn (from Kaggle env or utility script) ---
try:
    import flash_attn
    print(f"flash_attn={flash_attn.__version__}")
except ImportError:
    fa_whls = glob.glob("/kaggle/input/**/*flash_attn*.whl", recursive=True)
    if fa_whls:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", fa_whls[0]],
                       capture_output=True, text=True, timeout=120)
        print(f"Installed flash_attn from {os.path.basename(fa_whls[0])}")

# --- Verify key packages ---
print("\n=== Package Versions ===")
for pkg in ["unsloth", "unsloth_zoo", "trl", "peft", "datasets", "transformers"]:
    try:
        mod = __import__(pkg)
        print(f"  {pkg}={getattr(mod, '__version__', '?')}")
    except ImportError as e:
        if pkg == "unsloth":
            raise RuntimeError(f"FATAL: unsloth not importable: {e}")
        print(f"  WARNING: {pkg}: {e}")

print("\nPackage setup complete")


Found offline packages at: /kaggle/input/datasets/hastws/sft-offline-packages (9 wheels)
Installing 9 wheels with --no-deps...
  OK: annotated_doc-0.0.4-py3-none-any.whl
  OK: bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl
  OK: hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
  OK: nest_asyncio-1.6.0-py3-none-any.whl
  OK: psutil-7.2.2-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl
  OK: trl-0.29.1-py3-none-any.whl
  OK: tyro-1.0.12-py3-none-any.whl
  OK: unsloth-2026.4.4-py3-none-any.whl
  OK: unsloth_zoo-2026.4.6-py3-none-any.whl


[unsloth.import_fixes|WARNING]Unsloth: torch==2.12.0.dev20260324+cu128 requires torchvision>=0.27.0, but found torchvision==0.26.0.dev20260324+cu128. Try updating torchvision via `pip install --upgrade "torchvision>=0.27.0"`. Please refer to https://pytorch.org/get-started/previous-versions/ for more information.
Detected a pre-release build. Continuing with a warning. Set UNSLOTH_SKIP_TORCHVISION_CHECK=1 to silence this.


flash_attn=2.8.3

=== Package Versions ===
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


🦥 Unsloth Zoo will now patch everything to make training faster!
  unsloth=2026.4.4
  unsloth_zoo=2026.4.6
  trl=0.29.1
  peft=0.18.1
  datasets=4.0.0
  transformers=5.3.0

Package setup complete


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys
import stat
import shutil
import gc
import zipfile
import time
import json
import math
import re
import polars as pl
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset

# NOTE: trl imports moved to training cell (after Unsloth import)
# to ensure Unsloth's monkey-patching takes effect

print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"GPU memory: {mem / 1024**3:.1f} GB")

# --- Triton / Blackwell environment fixes ---
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# Copy PTXAS binaries to writable temp — search multiple paths
_ptxas_candidates = [
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
]
# Also search recursively
import glob as _g
_ptxas_candidates += _g.glob("/kaggle/usr/**/ptxas*blackwell", recursive=True)
_ptxas_candidates += _g.glob("/kaggle/input/**/ptxas*blackwell", recursive=True)

src = None
for _candidate in _ptxas_candidates:
    if os.path.exists(_candidate):
        src = _candidate
        break

dst = "/tmp/ptxas-blackwell"
if src:
    print(f"Found ptxas-blackwell at: {src}")
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton patched for Blackwell")
else:
    print("ptxas-blackwell not found, skipping Triton patch")

print("Imports and environment ready")

torch: 2.12.0.dev20260324+cu128, CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 95.0 GB
Found ptxas-blackwell at: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell
Triton patched for Blackwell
Imports and environment ready


## Configuration

Key changes from V123 (HF PEFT):
- `alpha=32` (was 64) — aligned with 0.85 solution
- `dropout=0.0` (was 0.05) — no regularization
- `target_modules="all-linear"` — Unsloth string mode covers gate_proj, x_proj, experts, lm_head

In [3]:
# =============================================
#  HYPERPARAMETERS
# =============================================

# --- Training ---
STAGE1_LR        = 1e-4
STAGE1_EPOCHS    = 2
STAGE1_MAX_SEQ   = 2048
STAGE1_BATCH     = 1
STAGE1_GRAD_ACCUM = 4
STAGE1_PACKING   = False
STAGE1_ANSWER_ONLY = False

# --- LoRA (Unsloth) ---
LORA_RANK        = 32
LORA_ALPHA       = 32       # Aligned with 0.85 (was 64)
LORA_DROPOUT     = 0.0      # Aligned with 0.85 (was 0.05)

# --- Boxed Loss Weight ---
BOXED_LOSS_WEIGHT = 5.0

# --- Type Filter ---
TRAIN_TYPES      = []

# --- Stage2 (disabled) ---
STAGE2_ENABLED   = False

# --- Holdout ---
HOLDOUT_ENABLED  = False
HOLDOUT_N_PER_TYPE = 10

# --- Prompt Suffix (official) ---
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)
EVAL_MAX_TOKENS  = 3584

print(f"Training: LR={STAGE1_LR}, epochs={STAGE1_EPOCHS}, max_seq={STAGE1_MAX_SEQ}")
print(f"Batch: {STAGE1_BATCH} x {STAGE1_GRAD_ACCUM} = {STAGE1_BATCH * STAGE1_GRAD_ACCUM}")
print(f"LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Boxed loss weight: {BOXED_LOSS_WEIGHT}")
print(f"Train types: {TRAIN_TYPES if TRAIN_TYPES else 'ALL'}")

Training: LR=0.0001, epochs=2, max_seq=2048
Batch: 1 x 4 = 4
LoRA: rank=32, alpha=32, dropout=0.0
Boxed loss weight: 5.0
Train types: ALL


## Data Loading

In [4]:
print("=== DATA LOADING ===")

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
COMP_DATA = '/kaggle/input/nvidia-nemotron-3-reasoning-challenge'

# --- Find training data ---
TARGET_FILE = 'sft_thinking.csv'
_COT_CANDIDATES = [
    '/kaggle/input/prog-cot-training-data',
    '/kaggle/input/datasets/hastws/prog-cot-training-data',
]
COT_DATA = None
for d in _COT_CANDIDATES:
    if os.path.isfile(os.path.join(d, TARGET_FILE)):
        COT_DATA = d
        break
if COT_DATA is None:
    import glob as _glob
    matches = _glob.glob(f'/kaggle/input/**/{TARGET_FILE}', recursive=True)
    if matches:
        COT_DATA = os.path.dirname(matches[0])
    else:
        raise FileNotFoundError(f"{TARGET_FILE} not found under /kaggle/input/")

print(f"COT_DATA = {COT_DATA}")
train_df = pl.read_csv(f'{COT_DATA}/{TARGET_FILE}')
print(f"Loaded: {len(train_df)} rows")

# Type distribution
pdf = train_df.to_pandas()
if 'type' in pdf.columns:
    print("\nType distribution:")
    for t in sorted(pdf['type'].unique()):
        print(f"  {t}: {(pdf['type'] == t).sum()}")

# --- Type inference ---
import re as _re_type
_NUM_EQ_RE = _re_type.compile(r'^(\d+)([^\d])(\d+)$')

def _is_numeric_equation(prompt):
    for line in prompt.strip().split('\n'):
        line = line.strip()
        if ' = ' in line and 'alice' not in line.lower() and 'equation' not in line.lower() \
                and 'transformation' not in line.lower() and 'determine' not in line.lower():
            lhs = line.split(' = ', 1)[0].strip()
            if _NUM_EQ_RE.match(lhs):
                return True
    return False

def _infer_type(prompt):
    p = prompt.lower()
    if 'bit manipulation' in p or '8-bit binary' in p:
        return 'bit_ops'
    elif 'numeral system' in p:
        return 'numeral'
    elif 'encrypt' in p or 'decrypt' in p:
        return 'cipher'
    elif 'gravitational' in p:
        return 'gravity'
    elif 'unit' in p and 'convert' in p:
        return 'unit_conv'
    elif 'symbol' in p or 'transformation rule' in p:
        return 'eq_numeric' if _is_numeric_equation(prompt) else 'eq_symbolic'
    return 'unknown'


=== DATA LOADING ===
COT_DATA = /kaggle/input/datasets/hastws/prog-cot-training-data
Loaded: 5200 rows

Type distribution:
  bit_ops: 1600
  cipher: 800
  eq_numeric: 800
  eq_symbolic: 800
  gravity: 400
  numeral: 400
  unit_conv: 400


## Format Training Text

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer vocab size: {len(tokenizer)}")

def _has_thinking(thinking):
    if thinking is None:
        return False
    if isinstance(thinking, float) and math.isnan(thinking):
        return False
    s = str(thinking).strip()
    return len(s) > 0 and s.lower() != 'nan'

def build_training_text(example):
    prompt = example["prompt"]
    answer = str(example["answer"])
    user_msg = prompt + PROMPT_SUFFIX

    if STAGE1_ANSWER_ONLY:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n<think></think>\\boxed{{{answer}}}<|im_end|>"
        )
    else:
        thinking = example.get("thinking", None)
        if _has_thinking(thinking):
            text = (
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n<think>\n{str(thinking).strip()}\n</think>\n"
                f"\\boxed{{{answer}}}<|im_end|>"
            )
        else:
            text = (
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n<think></think>\\boxed{{{answer}}}<|im_end|>"
            )
    return {"text": text}

# --- Build dataset ---
stage1_pdf = train_df.to_pandas()

# Type-based filtering
if TRAIN_TYPES:
    if 'type' not in stage1_pdf.columns:
        stage1_pdf['_type'] = stage1_pdf['prompt'].apply(_infer_type)
        stage1_pdf = stage1_pdf[stage1_pdf['_type'].isin(TRAIN_TYPES)].reset_index(drop=True)
    else:
        stage1_pdf = stage1_pdf[stage1_pdf['type'].isin(TRAIN_TYPES)].reset_index(drop=True)
    print(f"Type filter: {TRAIN_TYPES} -> {len(stage1_pdf)} rows")

# Holdout split
holdout_df = None
if HOLDOUT_ENABLED:
    type_col = stage1_pdf['type'] if 'type' in stage1_pdf.columns else stage1_pdf['prompt'].apply(_infer_type)
    holdout_parts = []
    for t in sorted(type_col.unique()):
        t_df = stage1_pdf[type_col == t]
        holdout_parts.append(t_df.sample(n=min(HOLDOUT_N_PER_TYPE, len(t_df)), random_state=42))
    holdout_df = pd.concat(holdout_parts).reset_index(drop=True)
    holdout_df['type'] = holdout_df['prompt'].apply(_infer_type)
    stage1_pdf = stage1_pdf.drop(holdout_df.index).reset_index(drop=True)
    print(f"Holdout: {len(holdout_df)}, Training: {len(stage1_pdf)}")

hf_dataset = Dataset.from_pandas(stage1_pdf)
hf_dataset = hf_dataset.map(build_training_text, remove_columns=hf_dataset.column_names)
print(f"Dataset: {len(hf_dataset)} rows")

# Token length analysis
token_lengths = []
for i in range(len(hf_dataset)):
    toks = tokenizer(hf_dataset[i]['text'], add_special_tokens=False)
    token_lengths.append(len(toks['input_ids']))
tl = np.array(token_lengths)
print(f"Token lengths: min={tl.min()}, median={np.median(tl):.0f}, max={tl.max()}")
print(f"  >{STAGE1_MAX_SEQ}: {(tl > STAGE1_MAX_SEQ).sum()} ({(tl > STAGE1_MAX_SEQ).mean()*100:.1f}%)")


Tokenizer vocab size: 131072


Map:   0%|          | 0/5200 [00:00<?, ? examples/s]

Dataset: 5200 rows
Token lengths: min=127, median=437, max=1734
  >2048: 0 (0.0%)


## Model Loading with Unsloth

Uses `FastLanguageModel` from Unsloth. This splits Mamba's `in_proj` into `gate_proj` + `x_proj`
with separate LoRA on each — more fine-grained than HF PEFT. Also fuses MoE expert weights for 3D LoRA.

In [ ]:
from unsloth import FastLanguageModel
from unittest.mock import MagicMock

# Mock problematic CUDA modules
_mock_modules = [
    "cutlass", "cutlass.cute", "cutlass.utils",
    "mamba_ssm.ops.cute", "mamba_ssm.ops.cute.mamba3",
    "mamba_ssm.ops.cute.mamba3.mamba3_step_fn",
    "mamba_ssm.ops.tilelang", "mamba_ssm.ops.tilelang.mamba3",
    "mamba_ssm.ops.tilelang.mamba3.mamba3_mimo",
]
for mod_name in _mock_modules:
    if mod_name not in sys.modules:
        sys.modules[mod_name] = MagicMock()

t0 = time.time()

# Load with Unsloth — use LOCAL metric/ model path (no internet needed!)
# Unsloth auto-detects NemotronH architecture and splits Mamba in_proj → gate_proj + x_proj
# Also fuses MoE expert weights (w1/w2) for efficient 3D LoRA
print(f"Loading model with Unsloth from: {MODEL_PATH}")
model, unsloth_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=STAGE1_MAX_SEQ,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    trust_remote_code=True,
    unsloth_force_compile=False,
    attn_implementation="eager",
)
print(f"Model loaded in {time.time()-t0:.1f}s")

# Patch: force slow path for Blackwell
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        if hasattr(mod, 'is_fast_path_available'):
            mod.is_fast_path_available = False
            print(f"Patched {name}: is_fast_path_available = False")

# Setup LoRA with Unsloth
print(f"\nLoRA: r={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
# Explicit target_modules to bypass Unsloth's get_peft_regex()
# which fails on metric/ models (backbone.layers vs model.layers regex mismatch).
# Includes ALL possible module types — PEFT auto-skips non-matching names.
# Unsloth's get_moe_target_parameters() auto-detects expert layers from this list.
# NOTE (V126): removed lm_head — vocab×d LoRA + gradient checkpointing was a major
# throughput killer (~2x slower per step) with negligible quality benefit.
_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # Attention
    "up_proj", "down_proj",                     # MoE experts
    "in_proj", "out_proj",                      # Mamba (original)
    "gate_proj", "x_proj",                      # Mamba (Unsloth-split)
]
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=_TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Print trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params/1e9:.2f}B")
print(f"Trainable params: {trainable_params/1e6:.1f}M ({trainable_params/total_params*100:.3f}%)")

# List LoRA modules for verification
print("\n=== LoRA Module Coverage ===")
_lora_modules = set()
for name, param in model.named_parameters():
    if param.requires_grad and 'lora' in name.lower():
        parts = name.split('.')
        for i, p in enumerate(parts):
            if 'lora' in p.lower():
                module_path = '.'.join(parts[max(0,i-2):i])
                _lora_modules.add(module_path)
                break
print(f"Unique LoRA target types: {len(_lora_modules)}")
for m in sorted(_lora_modules)[:20]:
    print(f"  {m}")
if len(_lora_modules) > 20:
    print(f"  ... and {len(_lora_modules) - 20} more")


Loading model with Unsloth from: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.4: Fast Nemotron_H patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0.dev20260324+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.
Model loaded in 470.1s
Patched transformers_modules._1.modeling_nemotron_h: is_fast_path_available = False

LoRA: r=32, alpha=32, dropout=0.0
Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'in_proj', 'out_proj', 'gate_proj', 'x_proj', 'lm_head']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.backbone` require gradients
Total params: 32.47B
Trainable params: 888.2M (2.736%)

=== LoRA Module Coverage ===
Unique LoRA target types: 265
  0.down_proj
  0.up_proj
  1.down_proj
  1.up_proj
  10.down_proj
  10.up_proj
  100.do

## Training with SFTTrainer + Boxed Loss Weight

In [ ]:
import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

# Import trl AFTER Unsloth to ensure monkey-patching takes effect
from trl import SFTTrainer, SFTConfig

eff_batch = STAGE1_BATCH * STAGE1_GRAD_ACCUM
total_steps = (len(hf_dataset) // eff_batch) * STAGE1_EPOCHS
print(f"{'='*60}")
print(f"  TRAINING: Unsloth + SFTTrainer")
print(f"  Samples: {len(hf_dataset)}, LR: {STAGE1_LR}, Epochs: {STAGE1_EPOCHS}")
print(f"  Max Seq: {STAGE1_MAX_SEQ}, Effective batch: {eff_batch}")
print(f"  Boxed loss weight: {BOXED_LOSS_WEIGHT}")
print(f"  Steps: ~{total_steps}")
print(f"{'='*60}")

# NOTE: gradient_checkpointing NOT set here — Unsloth handles it
# via use_gradient_checkpointing="unsloth" in get_peft_model (Cell 10).
# Setting both causes conflicts.
stage1_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=STAGE1_BATCH,
    gradient_accumulation_steps=STAGE1_GRAD_ACCUM,
    num_train_epochs=STAGE1_EPOCHS,
    learning_rate=STAGE1_LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=STAGE1_MAX_SEQ,
    packing=STAGE1_PACKING,
)

# ── Weighted Boxed Loss (V126: fully vectorized, no Python loop / no GPU sync) ──
_compute_loss_fn = None
if BOXED_LOSS_WEIGHT > 1.0:
    _boxed_marker_ids = tokenizer.encode("\\boxed{", add_special_tokens=False)
    _boxed_weight = float(BOXED_LOSS_WEIGHT)
    print(f"[loss] \\boxed{{ marker: {len(_boxed_marker_ids)} IDs: {_boxed_marker_ids}")

    _MARKER_TENSOR = torch.tensor(_boxed_marker_ids, dtype=torch.long)
    _MARKER_LEN = len(_boxed_marker_ids)

    def _weighted_boxed_loss(outputs, labels, num_items_in_batch=None):
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        batch_size, seq_len = shift_labels.shape
        device = shift_labels.device

        loss_fct = torch.nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
        per_token_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view(batch_size, seq_len)

        # Vectorized: find last position where marker appears in each row.
        # Build a (B, seq_len - m + 1) match mask via unfold without sync.
        marker = _MARKER_TENSOR.to(device)
        m = _MARKER_LEN
        if seq_len >= m:
            # windows: (B, seq_len - m + 1, m)
            windows = shift_labels.unfold(dimension=1, size=m, step=1)
            match = (windows == marker).all(dim=-1)  # (B, L')
            # Position index per match (1-based so 0 means no match)
            pos_idx = torch.arange(1, match.size(1) + 1, device=device)
            last_pos = (match * pos_idx).max(dim=1).values - 1  # (B,), -1 if none
            # Build weight mask: positions >= last_pos get _boxed_weight
            col = torch.arange(seq_len, device=device).unsqueeze(0)  # (1, seq_len)
            apply = (last_pos.unsqueeze(1) >= 0) & (col >= last_pos.unsqueeze(1))
            weights = torch.where(
                apply,
                torch.full_like(per_token_loss, _boxed_weight),
                torch.ones_like(per_token_loss),
            )
        else:
            weights = torch.ones_like(per_token_loss)

        mask = (shift_labels != -100).to(per_token_loss.dtype)
        denom = (weights * mask).sum().clamp_min(1.0)
        weighted_loss = (per_token_loss * weights * mask).sum() / denom
        return weighted_loss

    _compute_loss_fn = _weighted_boxed_loss
    print(f"[loss] Weighted boxed loss (vectorized): {_boxed_weight}x after last \\boxed{{")

stage1_trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=stage1_args,
    compute_loss_func=_compute_loss_fn,
)

t0 = time.time()
stage1_result = stage1_trainer.train()
stage1_time = time.time() - t0

print(f"\n{'='*60}")
print(f"  TRAINING COMPLETE")
print(f"  Time: {stage1_time/60:.1f} min")
print(f"  Final loss: {stage1_result.training_loss:.4f}")
print(f"{'='*60}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  TRAINING: Unsloth + SFTTrainer
  Samples: 5200, LR: 0.0001, Epochs: 2
  Max Seq: 2048, Effective batch: 4
  Boxed loss weight: 5.0
  Steps: ~2600
[loss] \boxed{ marker: 4 IDs: [1092, 4873, 1286, 1123]
[loss] Weighted boxed loss: 5.0x after last \boxed{


Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/5200 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,19.696466
20,20.056383
30,20.036449
40,17.856964
50,14.809453
60,13.851555
70,11.755311
80,8.820412
90,6.313354
100,5.460062



  TRAINING COMPLETE
  Time: 617.4 min
  Final loss: 1.7135


## Save & Convert Unsloth → HF PEFT

1. **Key rename**: `base_model.model.model` → `base_model.model.backbone`
2. **Expert unfuse**: `experts.w1` → `experts.{i}.up_proj`, `experts.w2` → `experts.{i}.down_proj`
3. **Mamba merge**: `gate_proj` + `x_proj` → `in_proj` via SVD (rank-64 → rank-32, ~98% preserved)
4. **Config rewrite**: explicit target_modules for HF PEFT

In [8]:
# --- Save raw Unsloth adapter ---
UNSLOTH_DIR = "/kaggle/working/unsloth_adapter"
os.makedirs(UNSLOTH_DIR, exist_ok=True)
model.save_pretrained(UNSLOTH_DIR)
print(f"Unsloth adapter saved to {UNSLOTH_DIR}")
for f in sorted(os.listdir(UNSLOTH_DIR)):
    size = os.path.getsize(os.path.join(UNSLOTH_DIR, f))
    print(f"  {f:40s} {size/1024:.1f} KB")

# --- Load metric/ model key shapes for conversion ---
from safetensors import safe_open
from safetensors.torch import save_file
import glob as _glob

model_keys = set()
for model_sf in _glob.glob(os.path.join(MODEL_PATH, "*.safetensors")):
    with safe_open(model_sf, framework="pt", device="cpu") as f:
        for key in f.keys():
            tensor_slice = f.get_slice(key)
            model_keys.add((key, tuple(tensor_slice.get_shape()), tensor_slice.get_dtype()))
print(f"Loaded {len(model_keys)} model key shapes from metric/ model")

# --- Key rename function ---
def trained_adapter_key_rename(key_name):
    return key_name.replace("base_model.model.model", "base_model.model.backbone")

# --- Convert Unsloth → HF PEFT ---
print("\n=== Converting Unsloth -> HF PEFT ===")

adapter_tensors = {}
with safe_open(os.path.join(UNSLOTH_DIR, "adapter_model.safetensors"), framework="pt", device="cpu") as f:
    for key in f.keys():
        adapter_tensors[key] = f.get_tensor(key)
print(f"Loaded {len(adapter_tensors)} adapter tensors")

# Collect base names
base_names = set()
for key in adapter_tensors:
    base = re.sub(r"\.lora_[AB]\.weight$", "", key)
    base_names.add(base)
print(f"Found {len(base_names)} LoRA base names")

# Identify Mamba layers needing gate_proj+x_proj -> in_proj merge
mamba_merge_layers = {}
for base in base_names:
    for proj in ("gate_proj", "x_proj"):
        if f".{proj}" in base:
            layer_path = base.rsplit(f".{proj}", 1)[0]
            mamba_merge_layers.setdefault(layer_path, {})[proj] = base
mamba_merge_bases = set()
for projs in mamba_merge_layers.values():
    mamba_merge_bases.update(projs.values())
print(f"Mamba merge: {len(mamba_merge_layers)} layers")

model_key_shapes = {k: s for k, s, _ in model_keys}

# Build converted tensors
tensors = {}

for base in sorted(base_names):
    lora_A = adapter_tensors[f"{base}.lora_A.weight"]
    lora_B = adapter_tensors[f"{base}.lora_B.weight"]
    renamed = trained_adapter_key_rename(base)

    # Skip empty w3 experts
    if ".experts.w3" in base and lora_A.numel() == 0:
        continue

    # Skip gate_proj/x_proj (handled in Mamba merge below)
    if base in mamba_merge_bases:
        continue

    # Expert unfusing: w1 -> per-expert up_proj, w2 -> per-expert down_proj
    if ".experts.w1" in base or ".experts.w2" in base:
        if lora_A.shape[0] == 1:
            lora_A = lora_A.expand(lora_B.shape[0], -1, -1).contiguous()
        elif lora_B.shape[0] == 1:
            lora_B = lora_B.expand(lora_A.shape[0], -1, -1).contiguous()

        num_experts = lora_A.shape[0]
        proj_name = "up_proj" if ".w1" in base else "down_proj"

        for i in range(num_experts):
            exp_renamed = re.sub(
                r"\.experts\.w[12]",
                f".experts.{i}.{proj_name}",
                renamed,
            )
            tensors[f"{exp_renamed}.lora_A.weight"] = lora_A[i].contiguous()
            tensors[f"{exp_renamed}.lora_B.weight"] = lora_B[i].contiguous()
        continue

    # Direct rename
    tensors[f"{renamed}.lora_A.weight"] = lora_A
    tensors[f"{renamed}.lora_B.weight"] = lora_B

# Mamba: gate_proj + x_proj -> in_proj via SVD
print("\nMamba gate_proj + x_proj -> in_proj SVD merge:")
for layer_path, projs in sorted(mamba_merge_layers.items()):
    renamed_layer = trained_adapter_key_rename(layer_path)
    in_proj_base = f"{renamed_layer}.in_proj"

    model_in_proj_key = (
        renamed_layer.replace("base_model.model.", "") + ".in_proj.weight"
    )
    in_proj_dim = model_key_shapes[model_in_proj_key][0]

    gate_A = adapter_tensors[f"{projs['gate_proj']}.lora_A.weight"].float()
    gate_B = adapter_tensors[f"{projs['gate_proj']}.lora_B.weight"].float()
    x_A = adapter_tensors[f"{projs['x_proj']}.lora_A.weight"].float()
    x_B = adapter_tensors[f"{projs['x_proj']}.lora_B.weight"].float()
    rank = gate_A.shape[0]

    A_cat = torch.cat([gate_A, x_A], dim=0)
    B_block = torch.zeros(in_proj_dim, 2 * rank)
    B_block[:gate_B.shape[0], :rank] = gate_B
    B_block[gate_B.shape[0]:gate_B.shape[0] + x_B.shape[0], rank:] = x_B

    Q_B, R_B = torch.linalg.qr(B_block)
    Q_A, R_A = torch.linalg.qr(A_cat.T)
    core = R_B @ R_A.T
    U, S, Vh = torch.linalg.svd(core, full_matrices=False)

    k = rank
    new_B = (Q_B @ U[:, :k]) * S[:k].unsqueeze(0)
    new_A = Vh[:k, :] @ Q_A.T

    kept = S[:k].sum().item()
    total = S.sum().item()
    print(f"  {layer_path}: SVD {kept:.2f}/{total:.2f} ({kept/total*100:.1f}%)")

    tensors[f"{in_proj_base}.lora_A.weight"] = new_A
    tensors[f"{in_proj_base}.lora_B.weight"] = new_B

print(f"\nConverted {len(adapter_tensors)} -> {len(tensors)} tensors")
save_file(tensors, os.path.join(OUTPUT_DIR, "adapter_model.safetensors"))
print(f"Saved to {OUTPUT_DIR}/adapter_model.safetensors")


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Unsloth adapter saved to /kaggle/working/unsloth_adapter
  README.md                                5.2 KB
  adapter_config.json                      1.3 KB
  adapter_model.safetensors                4159242.0 KB
Loaded 6243 model key shapes from metric/ model

=== Converting Unsloth -> HF PEFT ===
Loaded 12011 adapter tensors
Found 6006 LoRA base names
Mamba merge: 0 layers


KeyError: 'base_model.model.lm_head.base_layer.weight.lora_A.weight'

In [ ]:
# --- Write HF PEFT adapter_config.json ---
adapter_config = {
    "alpha_pattern": {},
    "auto_mapping": None,
    "base_model_name_or_path": str(MODEL_PATH),
    "bias": "none",
    "fan_in_fan_out": False,
    "inference_mode": True,
    "init_lora_weights": True,
    "layer_replication": None,
    "layers_pattern": None,
    "layers_to_transform": None,
    "loftq_config": {},
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "megatron_config": None,
    "megatron_core": "megatron.core",
    "modules_to_save": None,
    "peft_type": "LORA",
    "r": LORA_RANK,
    "rank_pattern": {},
    "revision": None,
    "target_modules": [
        "k_proj", "o_proj", "in_proj", "q_proj",
        "up_proj", "v_proj", "down_proj", "out_proj",
    ],
    "task_type": "CAUSAL_LM",
    "use_dora": False,
    "use_rslora": False,
}

config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(config_path, "w") as f:
    json.dump(adapter_config, f, indent=2)
print(f"Wrote adapter_config.json")
print(f"  r={adapter_config['r']}, alpha={adapter_config['lora_alpha']}")
print(f"  target_modules={adapter_config['target_modules']}")

# --- Package submission.zip ---
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ["adapter_model.safetensors", "adapter_config.json"]:
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, fname)
            print(f"  Added {fname} ({os.path.getsize(fpath)/1024/1024:.1f} MB)")

print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB")

# Verify
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    assert "adapter_config.json" in names
    assert "adapter_model.safetensors" in names
    print(f"Contents: {names}")

print("submission.zip ready!")


## Holdout Evaluation (Optional)

In [ ]:
# =============================================
#  HOLDOUT EVALUATION — Per-Type Accuracy
# =============================================
import re as _re

eval_time = 0
results_df = None

# Save training metrics before cleanup
_s1_loss = stage1_result.training_loss if 'stage1_result' in dir() else -1
_s2_loss = stage2_result.training_loss if (STAGE2_ENABLED and 'stage2_result' in dir()) else -1

if HOLDOUT_ENABLED and holdout_df is not None:

    # --- Memory cleanup: free training state before inference ---
    print("Freeing training memory...")
    _g = globals()
    for _obj_name in ['stage1_trainer', 'stage2_trainer', 'stage1_result', 'stage2_result',
                       'hf_dataset', 'stage2_tokenized', 'stage2_raw', 'collator',
                       'stage1_args', 'stage2_args']:
        if _obj_name in _g:
            del _g[_obj_name]
    gc.collect()
    torch.cuda.empty_cache()
    _mem_free = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"GPU free memory after cleanup: {_mem_free:.1f} GB")

    # --- Merge LoRA into base model to save ~8GB VRAM ---
    print("Merging LoRA into base model...")
    model = model.merge_and_unload()
    gc.collect()
    torch.cuda.empty_cache()
    _mem_free2 = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"GPU free memory after merge: {_mem_free2:.1f} GB (saved {_mem_free2 - _mem_free:.1f} GB)")

    print(f"Holdout types: {sorted(holdout_df['type'].unique().tolist())}")

    # --- Official answer extraction (from nemotron-baseline-evaluation.ipynb) ---
    def extract_final_answer(text):
        if text is None:
            return 'NOT_FOUND'
        matches = _re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
        if matches:
            non_empty = [m.strip() for m in matches if m.strip()]
            if non_empty:
                return non_empty[-1]
            return matches[-1].strip()
        patterns = [
            r'The final answer is:\s*([^\n]+)',
            r'Final answer is:\s*([^\n]+)',
            r'Final answer\s*[:：]\s*([^\n]+)',
            r'final answer\s*[:：]\s*([^\n]+)',
        ]
        for pattern in patterns:
            matches = _re.findall(pattern, text, _re.IGNORECASE)
            if matches:
                return matches[-1].strip()
        matches = _re.findall(r'-?\d+(?:\.\d+)?', text)
        if matches:
            return matches[-1]
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        return lines[-1] if lines else 'NOT_FOUND'

    # --- Official verify (stricter than competition for honest holdout) ---
    import math as _math
    _BINARY_RE = _re.compile(r'^[01]+$')
    def eval_verify(stored_answer, predicted):
        stored_answer = str(stored_answer).strip()
        predicted = str(predicted).strip()
        # String exact match first
        if predicted.lower() == stored_answer.lower():
            return True
        # Binary strings (e.g. bit_ops "00111111"): require EXACT match only
        # Official eval has a bug where float("00111111") succeeds and tolerance
        # makes near-miss binary strings pass. We enforce strict matching here.
        if len(stored_answer) > 1 and _BINARY_RE.match(stored_answer):
            return False
        # Numeric tolerance for non-binary answers
        try:
            stored_num = float(stored_answer)
            predicted_num = float(predicted)
            return _math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
        except Exception:
            return False

    EVAL_MAX_TOKENS = EVAL_MAX_TOKENS if 'EVAL_MAX_TOKENS' in dir() else 1024

    eval_results = []
    t0_eval = time.time()

    print(f"\n{'='*60}")
    print(f"  HOLDOUT EVAL: {len(holdout_df)} samples")
    print(f"  Types: {sorted(holdout_df['type'].unique().tolist())}")
    print(f"  Decoding: greedy (temperature=0.0) -- matches official eval")
    print(f"  Max new tokens: {EVAL_MAX_TOKENS}")
    print(f"{'='*60}")

    try:
        for eval_idx, row in holdout_df.iterrows():
            prompt = row['prompt']
            answer = str(row['answer'])
            qtype = row['type']

            # Build prompt exactly like official eval
            user_content = prompt + PROMPT_SUFFIX
            chat_prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_content}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )

            inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
            prompt_len = inputs['input_ids'].shape[1]

            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=EVAL_MAX_TOKENS,
                    do_sample=False,
                )

            gen_ids = output[0][prompt_len:]
            generated = tokenizer.decode(gen_ids, skip_special_tokens=False)
            predicted = extract_final_answer(generated)
            correct = eval_verify(answer, predicted)

            # --- Verbose per-sample output ---
            n_done = len(eval_results) + 1
            status = "✅" if correct else "❌"
            print(f"\n{'='*60}")
            print(f"  [{n_done}/{len(holdout_df)}] {status} {qtype} | expected={answer} | predicted={predicted} | tokens={len(gen_ids)}")
            print(f"{'='*60}")
            print(f"  FULL OUTPUT ({len(generated)} chars):")
            print(generated)
            print(f"{'─'*60}")

            eval_results.append({
                'id': row['id'],
                'type': qtype,
                'answer': answer,
                'predicted': predicted,
                'correct': correct,
                'gen_tokens': len(gen_ids),
                'generated': generated,
            })

            # Free KV cache every sample
            del output, inputs, gen_ids
            torch.cuda.empty_cache()

    except Exception as _eval_err:
        print(f"\n⚠️ Eval stopped after {len(eval_results)}/{len(holdout_df)} samples: {_eval_err}")
        print("Adapter was already saved — no data loss.")

    eval_time = time.time() - t0_eval

    # --- Results table (even if partial) ---
    if eval_results:
        import pandas as pd
        results_df = pd.DataFrame(eval_results)

        print(f"\n{'='*60}")
        _label = "PARTIAL" if len(eval_results) < len(holdout_df) else "FULL"
        print(f"  📊 HOLDOUT RESULTS [{_label}] (eval time: {eval_time/60:.1f} min)")
        print(f"{'='*60}")
        print(f"  {'Type':12s} {'Correct':>8s} {'Total':>6s} {'Acc':>8s} {'Avg Tokens':>11s}")
        print(f"  {'-'*47}")

        for t in sorted(results_df['type'].unique()):
            t_df = results_df[results_df['type'] == t]
            n_correct = t_df['correct'].sum()
            n_total = len(t_df)
            avg_tok = t_df['gen_tokens'].mean()
            acc = n_correct / n_total * 100
            bar = '█' * int(acc / 5) + '░' * (20 - int(acc / 5))
            print(f"  {t:12s} {n_correct:5d}/{n_total:<3d}   {acc:5.1f}%  {avg_tok:8.0f}  {bar}")

        total_correct = results_df['correct'].sum()
        total_n = len(results_df)
        total_acc = total_correct / total_n * 100
        print(f"  {'-'*47}")
        print(f"  {'TOTAL':12s} {total_correct:5d}/{total_n:<3d}   {total_acc:5.1f}%")

        # --- Show failures ---
        failures = results_df[~results_df['correct']]
        if len(failures) > 0:
            print(f"\n  ❌ Failures ({len(failures)} total, showing first 10):")
            for _, f in failures.head(10).iterrows():
                print(f"    [{f['type']:10s}] expected='{f['answer']}', got='{f['predicted']}' ({f['gen_tokens']} tok)")

        # --- Token stats ---
        print(f"\n  Token stats: min={results_df['gen_tokens'].min()}, "
              f"median={results_df['gen_tokens'].median():.0f}, "
              f"max={results_df['gen_tokens'].max()}")
        not_found = (results_df['predicted'] == 'NOT_FOUND').sum()
        if not_found > 0:
            print(f"  ⚠️ NOT_FOUND: {not_found} samples (no \\boxed{{}} in output)")
    else:
        print("No eval results collected.")

else:
    print("Holdout evaluation: SKIPPED (HOLDOUT_ENABLED=False)")

# =============================================
#  TRAINING SUMMARY
# =============================================
print("\n" + "=" * 60)
print("  TRAINING SUMMARY")
print("=" * 60)
print(f"  Stage 1: loss={_s1_loss:.4f}, time={stage1_time/60:.1f}min")
if STAGE2_ENABLED:
    print(f"  Stage 2: {n_sample} samples, loss={_s2_loss:.4f}, time={stage2_time/60:.1f}min")
    print(f"  Total training time: {(stage1_time + stage2_time)/60:.1f} min")
else:
    print(f"  Stage 2: SKIPPED")
    print(f"  Total time: {stage1_time/60:.1f} min")
print(f"  LoRA rank: {LORA_RANK}, alpha: {LORA_ALPHA}")
print(f"  Prompt suffix verified: ✅")
print(f"  Output: /kaggle/working/submission.zip")

if HOLDOUT_ENABLED and results_df is not None and len(results_df) > 0:
    total_correct = results_df['correct'].sum()
    total_n = len(results_df)
    print(f"  Holdout: {total_correct}/{total_n} = {total_correct/total_n*100:.1f}%")
    print(f"  Eval time: {eval_time/60:.1f} min")

print("=" * 60)